In [ ]:
import pandas as pd
import numpy as np
import sys
import io
import warnings
warnings.filterwarnings('ignore')


# LOAD & INSPECT DATA
df = pd.read_csv("Nassau Candy Distributor.csv")

print("=" * 80)
print("NASSAU CANDY DISTRIBUTOR — SHIPPING ROUTE EFFICIENCY ANALYSIS")
print("=" * 80)

print(f"\n Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nBasic Statistics:\n{df.describe()}")



NASSAU CANDY DISTRIBUTOR — SHIPPING ROUTE EFFICIENCY ANALYSIS

 Dataset Shape: 10194 rows × 18 columns

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost']

Data Types:
Row ID              int64
Order ID           object
Order Date         object
Ship Date          object
Ship Mode          object
Customer ID         int64
Country/Region     object
City               object
State/Province     object
Postal Code        object
Division           object
Region             object
Product ID         object
Product Name       object
Sales             float64
Units               int64
Gross Profit      float64
Cost              float64
dtype: object

Missing Values:
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City

In [ ]:
#DATA CLEANING & FEATURE ENGINEERING
# Parse dates (DD-MM-YYYY format)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y', dayfirst=True)

# Shipping delay in days
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Extract time components
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Quarter'] = df['Order Date'].dt.quarter

# Create Route identifier (Region → State → City)
df['Route'] = df['Region'] + ' → ' + df['State/Province'] + ' → ' + df['City']
df['Route_Region_State'] = df['Region'] + ' → ' + df['State/Province']

# Profit margin
df['Profit Margin (%)'] = (df['Gross Profit'] / df['Sales'] * 100).round(2)

# Cost per unit
df['Cost Per Unit'] = (df['Cost'] / df['Units']).round(2)

print("\n" + "=" * 80)
print("2. FEATURE ENGINEERING COMPLETE")
print("=" * 80)
print(f"  - Shipping Days range: {df['Shipping Days'].min()} to {df['Shipping Days'].max()} days")
print(f"  - Average Shipping Days: {df['Shipping Days'].mean():.1f} days")
print(f"  - Median Shipping Days: {df['Shipping Days'].median():.1f} days")
print(f"  - Unique Routes: {df['Route'].nunique()}")
print(f"  - Unique Region→State Routes: {df['Route_Region_State'].nunique()}")
print(f"  - Order Date Range: {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")
print(f"  - Ship Date Range: {df['Ship Date'].min().date()} to {df['Ship Date'].max().date()}")

# 3. SHIPPING MODE ANALYSIS
print("\n" + "=" * 80)
print("3. SHIPPING MODE ANALYSIS")
print("=" * 80)

ship_mode_stats = df.groupby('Ship Mode').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Median_Shipping_Days=('Shipping Days', 'median'),
    Min_Shipping_Days=('Shipping Days', 'min'),
    Max_Shipping_Days=('Shipping Days', 'max'),
    Std_Shipping_Days=('Shipping Days', 'std'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum'),
    Avg_Units=('Units', 'mean')
).round(2)
ship_mode_stats['Pct_of_Orders'] = (ship_mode_stats['Order_Count'] / len(df) * 100).round(2)

print(ship_mode_stats.to_string())




2. FEATURE ENGINEERING COMPLETE
  - Shipping Days range: 904 to 1642 days
  - Average Shipping Days: 1320.8 days
  - Median Shipping Days: 1274.0 days
  - Unique Routes: 616
  - Unique Region→State Routes: 59
  - Order Date Range: 2024-01-02 to 2025-12-31
  - Ship Date Range: 2026-06-30 to 2030-06-28

3. SHIPPING MODE ANALYSIS
                Order_Count  Avg_Shipping_Days  Median_Shipping_Days  Min_Shipping_Days  Max_Shipping_Days  Std_Shipping_Days  Total_Sales  Total_Profit  Avg_Units  Pct_of_Orders
Ship Mode                                                                                                                                                                         
First Class            1548            1338.28                1272.0                905               1638             265.63     21319.39      14011.09       3.70          15.19
Same Day                547            1333.44                1269.0                904               1636             253.81      71

In [ ]:

# 4. REGIONAL PERFORMANCE ANALYSIS
print("\n" + "=" * 80)
print("4. REGIONAL PERFORMANCE ANALYSIS")
print("=" * 80)

region_stats = df.groupby('Region').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Median_Shipping_Days=('Shipping Days', 'median'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum'),
    Unique_Customers=('Customer ID', 'nunique'),
    Unique_Cities=('City', 'nunique'),
    Unique_States=('State/Province', 'nunique')
).round(2)
region_stats['Avg_Profit_Margin'] = (region_stats['Total_Profit'] / region_stats['Total_Sales'] * 100).round(2)
region_stats = region_stats.sort_values('Avg_Shipping_Days', ascending=False)

print(region_stats.to_string())

# STATE-LEVEL SHIPPING PERFORMANCE
print("\n" + "=" * 80)
print("5. TOP 15 STATES BY ORDER VOLUME & THEIR SHIPPING PERFORMANCE")
print("=" * 80)

state_stats = df.groupby('State/Province').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Median_Shipping_Days=('Shipping Days', 'median'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum'),
    Region=('Region', 'first')
).round(2)
state_stats = state_stats.sort_values('Order_Count', ascending=False)

print("\nTop 15 States by Volume:")
print(state_stats.head(15).to_string())

print("\n\nTop 10 SLOWEST States (min 20 orders):")
slow_states = state_stats[state_stats['Order_Count'] >= 20].sort_values('Avg_Shipping_Days', ascending=False)
print(slow_states.head(10).to_string())

print("\n\nTop 10 FASTEST States (min 20 orders):")
fast_states = state_stats[state_stats['Order_Count'] >= 20].sort_values('Avg_Shipping_Days', ascending=True)
print(fast_states.head(10).to_string())




4. REGIONAL PERFORMANCE ANALYSIS
          Order_Count  Avg_Shipping_Days  Median_Shipping_Days  Total_Sales  Total_Profit  Unique_Customers  Unique_Cities  Unique_States  Avg_Profit_Margin
Region                                                                                                                                                      
Interior         2335            1323.09                1274.0     32037.60      21282.49              1176            182             14              66.43
Atlantic         2986            1322.75                1274.0     41197.24      26973.70              1435            115             20              65.47
Pacific          3253            1322.19                1274.0     46301.53      30485.94              1614            172             14              65.84
Gulf             1620            1311.37                1274.0     22247.26      14700.67               822            125             11              66.08

5. TOP 15 STATES BY ORD

In [ ]:
# ============================================================
# 6. ROUTE EFFICIENCY ANALYSIS
# ============================================================
print("\n" + "=" * 80)
print("6. ROUTE EFFICIENCY ANALYSIS (Region → State)")
print("=" * 80)

route_stats = df.groupby('Route_Region_State').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Median_Shipping_Days=('Shipping Days', 'median'),
    Std_Shipping_Days=('Shipping Days', 'std'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum')
).round(2)
route_stats['Efficiency_Score'] = (100 - (route_stats['Avg_Shipping_Days'] / route_stats['Avg_Shipping_Days'].max()) * 100).round(2)
route_stats = route_stats.sort_values('Avg_Shipping_Days', ascending=False)

print("\nTop 15 LEAST Efficient Routes (Region→State):")
print(route_stats.head(15).to_string())

print("\n\nTop 15 MOST Efficient Routes (Region→State):")
print(route_stats.sort_values('Avg_Shipping_Days', ascending=True).head(15).to_string())

# ============================================================
# 7. CITY-LEVEL HOTSPOTS
# ============================================================
print("\n" + "=" * 80)
print("7. CITY-LEVEL HOTSPOTS (min 10 orders)")
print("=" * 80)

city_stats = df.groupby(['City', 'State/Province', 'Region']).agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum')
).round(2)
city_stats = city_stats[city_stats['Order_Count'] >= 10]

print("\nTop 15 Cities with LONGEST Avg Shipping Days:")
print(city_stats.sort_values('Avg_Shipping_Days', ascending=False).head(15).to_string())

print("\n\nTop 15 Cities with SHORTEST Avg Shipping Days:")
print(city_stats.sort_values('Avg_Shipping_Days', ascending=True).head(15).to_string())

print("\n\nTop 15 Cities by Order Volume:")
print(city_stats.sort_values('Order_Count', ascending=False).head(15).to_string())




6. ROUTE EFFICIENCY ANALYSIS (Region → State)

Top 15 LEAST Efficient Routes (Region→State):
                                 Order_Count  Avg_Shipping_Days  Median_Shipping_Days  Std_Shipping_Days  Total_Sales  Total_Profit  Efficiency_Score
Route_Region_State                                                                                                                                   
Atlantic → West Virginia                   4            1638.00                1639.0               2.00        63.97         43.89              0.00
Interior → North Dakota                    7            1637.86                1637.0               1.46       109.66         73.96              0.01
Pacific → Saskatchewan                     2            1457.00                1457.0             258.80        29.25         18.99             11.05
Interior → Manitoba                       12            1455.33                1455.0             191.14       138.25         90.27             11.15
Interi

In [ ]:
# ============================================================
# 8. PRODUCT DIVISION ANALYSIS
# ============================================================
print("\n" + "=" * 80)
print("8. PRODUCT DIVISION ANALYSIS")
print("=" * 80)

division_stats = df.groupby('Division').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum'),
    Avg_Units=('Units', 'mean'),
    Unique_Products=('Product ID', 'nunique')
).round(2)
division_stats['Profit_Margin'] = (division_stats['Total_Profit'] / division_stats['Total_Sales'] * 100).round(2)

print(division_stats.to_string())

# ============================================================
# 9. SHIPPING DELAY DISTRIBUTION
# ============================================================
print("\n" + "=" * 80)
print("9. SHIPPING DELAY DISTRIBUTION")
print("=" * 80)

delay_bins = [0, 500, 700, 900, 1000, 1200, 1500, 2000]
delay_labels = ['0-500', '500-700', '700-900', '900-1000', '1000-1200', '1200-1500', '1500+']
df['Delay_Bucket'] = pd.cut(df['Shipping Days'], bins=delay_bins, labels=delay_labels, right=False)

delay_dist = df['Delay_Bucket'].value_counts().sort_index()
print("\nShipping Days Distribution:")
for bucket, count in delay_dist.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct)
    print(f"  {bucket:>12s}: {count:>5d} ({pct:5.1f}%) {bar}")




8. PRODUCT DIVISION ANALYSIS
           Order_Count  Avg_Shipping_Days  Total_Sales  Total_Profit  Avg_Units  Unique_Products  Profit_Margin
Division                                                                                                       
Chocolate         9844            1321.17    131692.90      88824.62       3.79                5          67.45
Other              310            1306.03      9663.25       4333.45       4.01                3          44.84
Sugar               40            1355.65       427.48        284.73       3.42                7          66.61

9. SHIPPING DELAY DISTRIBUTION

Shipping Days Distribution:
         0-500:     0 (  0.0%) 
       500-700:     0 (  0.0%) 
       700-900:     0 (  0.0%) 
      900-1000:  2051 ( 20.1%) ████████████████████
     1000-1200:     0 (  0.0%) 
     1200-1500:  4764 ( 46.7%) ██████████████████████████████████████████████
         1500+:  3379 ( 33.1%) █████████████████████████████████


In [ ]:
# TEMPORAL TRENDS
print("\n" + "=" * 80)
print("10. TEMPORAL TRENDS")
print("=" * 80)

yearly_stats = df.groupby('Order Year').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum')
).round(2)
print("\nYearly Performance:")
print(yearly_stats.to_string())

quarterly_stats = df.groupby(['Order Year', 'Order Quarter']).agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Total_Sales=('Sales', 'sum')
).round(2)
print("\n\nQuarterly Performance:")
print(quarterly_stats.to_string())

# SHIP MODE × REGION CROSS ANALYSIS
print("\n" + "=" * 80)
print("11. SHIP MODE × REGION CROSS ANALYSIS (Avg Shipping Days)")
print("=" * 80)

cross = df.pivot_table(
    values='Shipping Days',
    index='Region',
    columns='Ship Mode',
    aggfunc='mean'
).round(1)
print(cross.to_string())

cross_count = df.pivot_table(
    values='Row ID',
    index='Region',
    columns='Ship Mode',
    aggfunc='count'
)
print("\n\nOrder Count by Ship Mode × Region:")
print(cross_count.to_string())

# COUNTRY ANALYSIS
print("\n" + "=" * 80)
print("12. COUNTRY/REGION ANALYSIS")
print("=" * 80)

country_stats = df.groupby('Country/Region').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Gross Profit', 'sum')
).round(2)
print(country_stats.to_string())




10. TEMPORAL TRENDS

Yearly Performance:
            Order_Count  Avg_Shipping_Days  Total_Sales  Total_Profit
Order Year                                                           
2024               4181            1094.02     57956.20      38151.43
2025               6013            1478.56     83827.43      55291.37


Quarterly Performance:
                          Order_Count  Avg_Shipping_Days  Total_Sales
Order Year Order Quarter                                             
2024       1                      558            1084.46      7612.67
           2                      851            1100.26     11458.50
           3                     1171            1094.10     16255.13
           4                     1601            1093.97     22629.90
2025       1                      857            1491.60     11465.99
           2                     1302            1468.82     17693.05
           3                     1668            1474.37     23255.37
           4           

In [ ]:
# KEY FINDINGS SUMMARY
print(" KEY FINDINGS SUMMARY")


print(f"""
 DATASET OVERVIEW:
  • {df.shape[0]:,} shipping records across {df['Order Year'].nunique()} years
  • {df['Region'].nunique()} regions, {df['State/Province'].nunique()} states/provinces, {df['City'].nunique()} cities
  • {df['Customer ID'].nunique()} unique customers
  • {df['Product ID'].nunique()} unique products in {df['Division'].nunique()} divisions
  • {df['Ship Mode'].nunique()} shipping modes used
  • Countries: {', '.join(df['Country/Region'].unique())}

 SHIPPING PERFORMANCE:
  • Average shipping time: {df['Shipping Days'].mean():.1f} days
  • Median shipping time: {df['Shipping Days'].median():.1f} days
  • Shipping time range: {df['Shipping Days'].min()} to {df['Shipping Days'].max()} days
  • Standard deviation: {df['Shipping Days'].std():.1f} days

 FINANCIAL OVERVIEW:
  • Total Sales: ${df['Sales'].sum():,.2f}
  • Total Gross Profit: ${df['Gross Profit'].sum():,.2f}
  • Total Cost: ${df['Cost'].sum():,.2f}
  • Overall Profit Margin: {(df['Gross Profit'].sum()/df['Sales'].sum()*100):.1f}%
  • Total Units Shipped: {df['Units'].sum():,}

 REGIONAL INSIGHTS:
  • Most orders from: {region_stats['Order_Count'].idxmax()} ({region_stats['Order_Count'].max():,} orders)
  • Slowest region: {region_stats.index[0]} ({region_stats['Avg_Shipping_Days'].iloc[0]:.1f} avg days)
  • Fastest region: {region_stats.index[-1]} ({region_stats['Avg_Shipping_Days'].iloc[-1]:.1f} avg days)

 TOP PRODUCT:
  • Most ordered product: {df['Product Name'].value_counts().index[0]} ({df['Product Name'].value_counts().iloc[0]:,} orders)
""")



 KEY FINDINGS SUMMARY

 DATASET OVERVIEW:
  • 10,194 shipping records across 2 years
  • 4 regions, 59 states/provinces, 542 cities
  • 5044 unique customers
  • 15 unique products in 3 divisions
  • 4 shipping modes used
  • Countries: United States, Canada

 SHIPPING PERFORMANCE:
  • Average shipping time: 1320.8 days
  • Median shipping time: 1274.0 days
  • Shipping time range: 904 to 1642 days
  • Standard deviation: 262.4 days

 FINANCIAL OVERVIEW:
  • Total Sales: $141,783.63
  • Total Gross Profit: $93,442.80
  • Total Cost: $48,340.83
  • Overall Profit Margin: 65.9%
  • Total Units Shipped: 38,654

 REGIONAL INSIGHTS:
  • Most orders from: Pacific (3,253 orders)
  • Slowest region: Interior (1323.1 avg days)
  • Fastest region: Gulf (1311.4 avg days)

 TOP PRODUCT:
  • Most ordered product: Wonka Bar - Milk Chocolate (2,137 orders)



In [ ]:
# Feature Engineering

# Calculate Shipping Lead Time (days)
# Categorize routes by Factory → Customer Region
# Categorize routes by Factory → Customer State
# Group shipments by Ship Mode

df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d-%m-%Y', dayfirst=True)

# CALCULATE SHIPPING LEAD TIME (DAYS)
df['Shipping_Lead_Time_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Classify lead-time into speed tiers
def classify_lead_time(days):
    if days <= 365:
        return 'Fast (≤1 Year)'
    elif days <= 730:
        return 'Standard (1–2 Years)'
    elif days <= 900:
        return 'Slow (2–2.5 Years)'
    else:
        return 'Very Slow (>2.5 Years)'

df['Lead_Time_Category'] = df['Shipping_Lead_Time_Days'].apply(classify_lead_time)

print("\n Shipping Lead Time (Days)")
print(f"  Min Lead Time       : {df['Shipping_Lead_Time_Days'].min()} days")
print(f"  Max Lead Time       : {df['Shipping_Lead_Time_Days'].max()} days")
print(f"  Mean Lead Time      : {df['Shipping_Lead_Time_Days'].mean():.2f} days")
print(f"  Median Lead Time    : {df['Shipping_Lead_Time_Days'].median():.0f} days")
print(f"  Std Dev             : {df['Shipping_Lead_Time_Days'].std():.2f} days")

print("\n  Lead Time Category Distribution:")
lt_dist = df['Lead_Time_Category'].value_counts()
for cat, count in lt_dist.items():
    pct = count / len(df) * 100
    print(f"    {cat:<30s} : {count:>5,} orders ({pct:5.1f}%)")

# CATEGORIZE ROUTES: FACTORY → CUSTOMER REGION
# Nassau Candy is the single factory/distributor origin
FACTORY_NAME = "Nassau Candy (Factory)"

df['Route_Factory_to_Region'] = FACTORY_NAME + ' → ' + df['Region']

print("\n Route Categorization: Factory → Customer Region")


route_region = df.groupby('Route_Factory_to_Region').agg(
    Shipment_Count     = ('Row ID', 'count'),
    Avg_Lead_Time_Days = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time   = ('Shipping_Lead_Time_Days', 'median'),
    Min_Lead_Time      = ('Shipping_Lead_Time_Days', 'min'),
    Max_Lead_Time      = ('Shipping_Lead_Time_Days', 'max'),
    Total_Sales        = ('Sales', 'sum'),
    Total_Profit       = ('Gross Profit', 'sum'),
    Unique_Customers   = ('Customer ID', 'nunique'),
    Unique_States      = ('State/Province', 'nunique'),
    Unique_Cities      = ('City', 'nunique')
).round(2)

route_region['Pct_of_Total_Shipments'] = (route_region['Shipment_Count'] / len(df) * 100).round(2)
route_region['Avg_Profit_Margin_%']    = (route_region['Total_Profit'] / route_region['Total_Sales'] * 100).round(2)
route_region = route_region.sort_values('Shipment_Count', ascending=False)

print(route_region.to_string())

# CATEGORIZE ROUTES: FACTORY → CUSTOMER STATE
df['Route_Factory_to_State'] = FACTORY_NAME + ' → ' + df['State/Province']

print("\n Route Categorization: Factory → Customer State")


route_state = df.groupby('Route_Factory_to_State').agg(
    Shipment_Count     = ('Row ID', 'count'),
    Avg_Lead_Time_Days = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time   = ('Shipping_Lead_Time_Days', 'median'),
    Total_Sales        = ('Sales', 'sum'),
    Total_Profit       = ('Gross Profit', 'sum'),
    Region             = ('Region', 'first'),
    Unique_Cities      = ('City', 'nunique'),
    Unique_Customers   = ('Customer ID', 'nunique')
).round(2)

route_state['Pct_of_Total_Shipments'] = (route_state['Shipment_Count'] / len(df) * 100).round(2)
route_state['Avg_Profit_Margin_%']    = (route_state['Total_Profit'] / route_state['Total_Sales'] * 100).round(2)
route_state = route_state.sort_values('Shipment_Count', ascending=False)

print("\n  Top 20 States by Shipment Volume:")
print(route_state.head(20).to_string())

print(f"\n  Total unique Factory → State routes: {route_state.shape[0]}")

print("\n\n  Top 10 Slowest Factory → State Routes (min 20 shipments):")
slow_routes = route_state[route_state['Shipment_Count'] >= 20].sort_values('Avg_Lead_Time_Days', ascending=False)
print(slow_routes.head(10).to_string())

print("\n\n  Top 10 Fastest Factory → State Routes (min 20 shipments):")
fast_routes = route_state[route_state['Shipment_Count'] >= 20].sort_values('Avg_Lead_Time_Days', ascending=True)
print(fast_routes.head(10).to_string())

# GROUP SHIPMENTS BY SHIP MODE
print("\n Shipments Grouped by Ship Mode")

ship_mode_group = df.groupby('Ship Mode').agg(
    Shipment_Count     = ('Row ID', 'count'),
    Avg_Lead_Time_Days = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time   = ('Shipping_Lead_Time_Days', 'median'),
    Min_Lead_Time      = ('Shipping_Lead_Time_Days', 'min'),
    Max_Lead_Time      = ('Shipping_Lead_Time_Days', 'max'),
    Std_Lead_Time      = ('Shipping_Lead_Time_Days', 'std'),
    Total_Sales        = ('Sales', 'sum'),
    Total_Profit       = ('Gross Profit', 'sum'),
    Total_Units        = ('Units', 'sum'),
    Avg_Units_Per_Order= ('Units', 'mean'),
    Unique_Customers   = ('Customer ID', 'nunique'),
    Unique_Regions     = ('Region', 'nunique'),
    Unique_States      = ('State/Province', 'nunique')
).round(2)

ship_mode_group['Pct_of_Total_Shipments'] = (ship_mode_group['Shipment_Count'] / len(df) * 100).round(2)
ship_mode_group['Avg_Profit_Margin_%']    = (ship_mode_group['Total_Profit'] / ship_mode_group['Total_Sales'] * 100).round(2)
ship_mode_group['Revenue_Per_Shipment']   = (ship_mode_group['Total_Sales'] / ship_mode_group['Shipment_Count']).round(2)

print(ship_mode_group.to_string())

# --- Ship Mode × Region Breakdown ---
print("\n\n  Ship Mode × Region — Average Lead Time (Days):")
cross_lt = df.pivot_table(
    values='Shipping_Lead_Time_Days',
    index='Region',
    columns='Ship Mode',
    aggfunc='mean'
).round(1)
print(cross_lt.to_string())

print("\n\n  Ship Mode × Region — Shipment Count:")
cross_cnt = df.pivot_table(
    values='Row ID',
    index='Region',
    columns='Ship Mode',
    aggfunc='count'
)
print(cross_cnt.to_string())

# --- Ship Mode × Lead Time Category ---
print("\n\n  Ship Mode × Lead Time Category — Distribution:")
cross_lt_cat = pd.crosstab(df['Ship Mode'], df['Lead_Time_Category'], margins=True)
print(cross_lt_cat.to_string())

# SUMMARY OF NEW ENGINEERED FEATURES
print("SUMMARY OF ENGINEERED FEATURES")


new_cols = ['Shipping_Lead_Time_Days', 'Lead_Time_Category',
            'Route_Factory_to_Region', 'Route_Factory_to_State']

print(f"\n  New columns added ({len(new_cols)}):")
for col in new_cols:
    print(f"    • {col:<30s}  |  dtype: {df[col].dtype}  |  unique: {df[col].nunique()}")

print(f"\n  Updated DataFrame shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  All columns: {list(df.columns)}")

# --- Preview ---
print("\n\n  Sample of engineered features (first 10 rows):")
print(df[['Order ID', 'Ship Mode', 'Order Date', 'Ship Date',
          'Shipping_Lead_Time_Days', 'Lead_Time_Category',
          'Route_Factory_to_Region', 'Route_Factory_to_State']].head(10).to_string(index=False))

print("FEATURE ENGINEERING COMPLETE")




 Shipping Lead Time (Days)
  Min Lead Time       : 904 days
  Max Lead Time       : 1642 days
  Mean Lead Time      : 1320.84 days
  Median Lead Time    : 1274 days
  Std Dev             : 262.44 days

  Lead Time Category Distribution:
    Very Slow (>2.5 Years)         : 10,194 orders (100.0%)

 Route Categorization: Factory → Customer Region
                                   Shipment_Count  Avg_Lead_Time_Days  Median_Lead_Time  Min_Lead_Time  Max_Lead_Time  Total_Sales  Total_Profit  Unique_Customers  Unique_States  Unique_Cities  Pct_of_Total_Shipments  Avg_Profit_Margin_%
Route_Factory_to_Region                                                                                                                                                                                                                      
Nassau Candy (Factory) → Pacific             3253             1322.19            1274.0            904           1642     46301.53      30485.94              1614             1

In [ ]:
#  Route Definition & Aggregation

# Each route = Factory Location → Customer State / Region
# For each route:
#   Total shipments
#   Average shipping lead time
#   Lead time variability (Std Dev, IQR, CV%)

df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d-%m-%Y', dayfirst=True)
df['Shipping_Lead_Time_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

FACTORY = "Nassau Candy (Factory)"

print("ROUTE DEFINITION & AGGREGATION")

# ROUTE DEFINITION
df['Route_to_Region'] = FACTORY + ' → ' + df['Region']
df['Route_to_State']  = FACTORY + ' → ' + df['State/Province']

print(f"\ Route Definitions Created:")
print(f"   Factory → Region routes : {df['Route_to_Region'].nunique()}")
print(f"   Factory → State routes  : {df['Route_to_State'].nunique()}")
print(f"   Total shipments          : {len(df):,}")

# AGGREGATION: FACTORY → CUSTOMER REGION
print("ROUTE AGGREGATION — Factory → Customer Region")

region_agg = df.groupby('Route_to_Region').agg(
    Total_Shipments      = ('Row ID', 'count'),
    Avg_Lead_Time        = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time     = ('Shipping_Lead_Time_Days', 'median'),
    Std_Dev_Lead_Time    = ('Shipping_Lead_Time_Days', 'std'),
    Min_Lead_Time        = ('Shipping_Lead_Time_Days', 'min'),
    Max_Lead_Time        = ('Shipping_Lead_Time_Days', 'max'),
    IQR_Lead_Time        = ('Shipping_Lead_Time_Days', lambda x: x.quantile(0.75) - x.quantile(0.25)),
    Q1_Lead_Time         = ('Shipping_Lead_Time_Days', lambda x: x.quantile(0.25)),
    Q3_Lead_Time         = ('Shipping_Lead_Time_Days', lambda x: x.quantile(0.75))
).round(2)

# Coefficient of Variation (CV%) — higher means more inconsistent
region_agg['CV_%'] = ((region_agg['Std_Dev_Lead_Time'] / region_agg['Avg_Lead_Time']) * 100).round(2)
region_agg['Pct_of_Total'] = (region_agg['Total_Shipments'] / len(df) * 100).round(2)
region_agg['Lead_Time_Range'] = region_agg['Max_Lead_Time'] - region_agg['Min_Lead_Time']
region_agg = region_agg.sort_values('Total_Shipments', ascending=False)

print("\n  Full Route Summary (Factory → Region):\n")
print(region_agg.to_string())

print("\n  Interpretation Guide:")
print("  Std_Dev_Lead_Time : How spread out lead times are (lower = more consistent)")
print("  IQR_Lead_Time     : Spread of middle 50% of shipments (robust to outliers)")
print("  CV_%              : Relative variability — comparable across routes")
print("  Lead_Time_Range   : Difference between fastest & slowest shipment")

# AGGREGATION: FACTORY → CUSTOMER STATE
print("ROUTE AGGREGATION — Factory → Customer State")

state_agg = df.groupby('Route_to_State').agg(
    Total_Shipments      = ('Row ID', 'count'),
    Avg_Lead_Time        = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time     = ('Shipping_Lead_Time_Days', 'median'),
    Std_Dev_Lead_Time    = ('Shipping_Lead_Time_Days', 'std'),
    Min_Lead_Time        = ('Shipping_Lead_Time_Days', 'min'),
    Max_Lead_Time        = ('Shipping_Lead_Time_Days', 'max'),
    IQR_Lead_Time        = ('Shipping_Lead_Time_Days', lambda x: x.quantile(0.75) - x.quantile(0.25)),
    Region               = ('Region', 'first')
).round(2)

state_agg['CV_%'] = ((state_agg['Std_Dev_Lead_Time'] / state_agg['Avg_Lead_Time']) * 100).round(2)
state_agg['Pct_of_Total'] = (state_agg['Total_Shipments'] / len(df) * 100).round(2)
state_agg['Lead_Time_Range'] = state_agg['Max_Lead_Time'] - state_agg['Min_Lead_Time']

# --- Top 20 by Volume ---
print("\n  Top 20 Routes by Shipment Volume:\n")
top_vol = state_agg.sort_values('Total_Shipments', ascending=False).head(20)
print(top_vol.to_string())

# --- Top 10 Slowest (min 20 shipments) ---
print("\n\n  Top 10 SLOWEST Routes (min 20 shipments):\n")
qualified = state_agg[state_agg['Total_Shipments'] >= 20]
slowest = qualified.sort_values('Avg_Lead_Time', ascending=False).head(10)
print(slowest[['Total_Shipments', 'Avg_Lead_Time', 'Median_Lead_Time',
               'Std_Dev_Lead_Time', 'CV_%', 'IQR_Lead_Time', 'Region']].to_string())

# --- Top 10 Fastest (min 20 shipments) ---
print("\n\n  Top 10 FASTEST Routes (min 20 shipments):\n")
fastest = qualified.sort_values('Avg_Lead_Time', ascending=True).head(10)
print(fastest[['Total_Shipments', 'Avg_Lead_Time', 'Median_Lead_Time',
               'Std_Dev_Lead_Time', 'CV_%', 'IQR_Lead_Time', 'Region']].to_string())

# --- Top 10 Most Variable / Inconsistent (min 20 shipments) ---
print("\n\n  Top 10 MOST VARIABLE Routes — Highest CV% (min 20 shipments):\n")
most_var = qualified.sort_values('CV_%', ascending=False).head(10)
print(most_var[['Total_Shipments', 'Avg_Lead_Time', 'Std_Dev_Lead_Time',
                'CV_%', 'IQR_Lead_Time', 'Lead_Time_Range', 'Region']].to_string())

# --- Top 10 Most Consistent (min 20 shipments) ---
print("\n\n  Top 10 MOST CONSISTENT Routes — Lowest CV% (min 20 shipments):\n")
most_con = qualified.sort_values('CV_%', ascending=True).head(10)
print(most_con[['Total_Shipments', 'Avg_Lead_Time', 'Std_Dev_Lead_Time',
                'CV_%', 'IQR_Lead_Time', 'Lead_Time_Range', 'Region']].to_string())

# HIGH VARIABILITY FLAG

print("HIGH VARIABILITY FLAGGING")

cv_threshold = state_agg['CV_%'].median()
state_agg['Variability_Flag'] = np.where(
    state_agg['CV_%'] > cv_threshold, ' High Variability', ' Consistent'
)

flag_summary = state_agg['Variability_Flag'].value_counts()
print(f"\n  CV% Threshold (median): {cv_threshold:.2f}%")
print(f"  Routes above threshold (High Variability): {flag_summary.get(' High Variability', 0)}")
print(f"  Routes below threshold (Consistent)      : {flag_summary.get(' Consistent', 0)}")

# COMBINED SUMMARY TABLE
print("COMBINED ROUTE SUMMARY TABLE")

summary = state_agg[['Region', 'Total_Shipments', 'Pct_of_Total',
                      'Avg_Lead_Time', 'Median_Lead_Time',
                      'Std_Dev_Lead_Time', 'IQR_Lead_Time', 'CV_%',
                      'Min_Lead_Time', 'Max_Lead_Time', 'Lead_Time_Range',
                      'Variability_Flag']].sort_values('Total_Shipments', ascending=False)

print(f"\n  All {summary.shape[0]} Factory → State Routes:\n")
print(summary.to_string())

#KEY TAKEAWAYS
print("KEY TAKEAWAYS")

overall_avg = df['Shipping_Lead_Time_Days'].mean()
overall_std = df['Shipping_Lead_Time_Days'].std()

busiest_route   = state_agg['Total_Shipments'].idxmax()
slowest_route   = qualified['Avg_Lead_Time'].idxmax()
fastest_route   = qualified['Avg_Lead_Time'].idxmin()
most_var_route  = qualified['CV_%'].idxmax()
most_con_route  = qualified['CV_%'].idxmin()

print(f"""
  TOTAL ROUTES DEFINED:
    • Factory → Region routes : {region_agg.shape[0]}
    • Factory → State routes  : {state_agg.shape[0]}

  OVERALL LEAD TIME:
    • Mean   : {overall_avg:.2f} days
    • Std Dev: {overall_std:.2f} days

  ROUTE HIGHLIGHTS (min 20 shipments):
    • Busiest Route       : {busiest_route} ({state_agg.loc[busiest_route, 'Total_Shipments']:,} shipments)
    • Slowest Avg Route   : {slowest_route} ({qualified.loc[slowest_route, 'Avg_Lead_Time']:.1f} days)
    • Fastest Avg Route   : {fastest_route} ({qualified.loc[fastest_route, 'Avg_Lead_Time']:.1f} days)
    • Most Variable Route : {most_var_route} (CV = {qualified.loc[most_var_route, 'CV_%']:.1f}%)
    • Most Consistent     : {most_con_route} (CV = {qualified.loc[most_con_route, 'CV_%']:.1f}%)

  VARIABILITY SUMMARY:
    • {flag_summary.get('High Variability', 0)} routes flagged as high variability (CV% > {cv_threshold:.1f}%)
    • {flag_summary.get('Consistent', 0)} routes are consistent
""")



ROUTE DEFINITION & AGGREGATION
\ Route Definitions Created:
   Factory → Region routes : 4
   Factory → State routes  : 59
   Total shipments          : 10,194
ROUTE AGGREGATION — Factory → Customer Region

  Full Route Summary (Factory → Region):

                                   Total_Shipments  Avg_Lead_Time  Median_Lead_Time  Std_Dev_Lead_Time  Min_Lead_Time  Max_Lead_Time  IQR_Lead_Time  Q1_Lead_Time  Q3_Lead_Time   CV_%  Pct_of_Total  Lead_Time_Range
Route_to_Region                                                                                                                                                                                                      
Nassau Candy (Factory) → Pacific              3253        1322.19            1274.0             266.47            904           1642          368.0        1270.0        1638.0  20.15         31.91              738
Nassau Candy (Factory) → Atlantic             2986        1322.75            1274.0             256.54       

In [ ]:
# EFFICIENCY BENCHMARKING
# Rank routes from fastest to slowest
# Identify Top 10 most efficient routes
# Identify Bottom 10 least efficient routes
# Compare performance across ship modes

df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d-%m-%Y', dayfirst=True)
df['Shipping_Lead_Time_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

FACTORY = "Nassau Candy (Factory)"
df['Route'] = FACTORY + ' → ' + df['State/Province']

print("EFFICIENCY BENCHMARKING")

# BUILD ROUTE EFFICIENCY TABLE
route_eff = df.groupby('Route').agg(
    Total_Shipments    = ('Row ID', 'count'),
    Avg_Lead_Time      = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time   = ('Shipping_Lead_Time_Days', 'median'),
    Std_Dev_Lead_Time  = ('Shipping_Lead_Time_Days', 'std'),
    Min_Lead_Time      = ('Shipping_Lead_Time_Days', 'min'),
    Max_Lead_Time      = ('Shipping_Lead_Time_Days', 'max'),
    Total_Sales        = ('Sales', 'sum'),
    Total_Profit       = ('Gross Profit', 'sum'),
    Region             = ('Region', 'first')
).round(2)

route_eff['CV_%'] = ((route_eff['Std_Dev_Lead_Time'] / route_eff['Avg_Lead_Time']) * 100).round(2)
route_eff['Profit_Margin_%'] = ((route_eff['Total_Profit'] / route_eff['Total_Sales']) * 100).round(2)

# Composite Efficiency Score
# Normalize: lower lead time = better, lower variability = better
# Score = 100 - weighted(normalized_avg + normalized_cv)
max_avg = route_eff['Avg_Lead_Time'].max()
min_avg = route_eff['Avg_Lead_Time'].min()
max_cv  = route_eff['CV_%'].max()
min_cv  = route_eff['CV_%'].min()

route_eff['Norm_Avg_Lead_Time'] = ((route_eff['Avg_Lead_Time'] - min_avg) / (max_avg - min_avg)).round(4)
route_eff['Norm_CV']            = ((route_eff['CV_%'] - min_cv) / (max_cv - min_cv)).round(4)

# Efficiency Score: 70% weight on speed, 30% on consistency (0–100 scale)
route_eff['Efficiency_Score'] = (100 - (0.70 * route_eff['Norm_Avg_Lead_Time'] +
                                         0.30 * route_eff['Norm_CV']) * 100).round(2)

# RANK ROUTES: FASTEST TO SLOWEST
print("ALL ROUTES RANKED — Fastest to Slowest")

route_ranked = route_eff.sort_values('Avg_Lead_Time', ascending=True).copy()
route_ranked['Speed_Rank'] = range(1, len(route_ranked) + 1)

# Also rank by efficiency score
route_eff_sorted = route_eff.sort_values('Efficiency_Score', ascending=False).copy()
route_eff_sorted['Efficiency_Rank'] = range(1, len(route_eff_sorted) + 1)

# Merge ranks
route_ranked['Efficiency_Rank'] = route_ranked.index.map(
    route_eff_sorted['Efficiency_Rank']
)

display_cols = ['Speed_Rank', 'Efficiency_Rank', 'Region', 'Total_Shipments',
                'Avg_Lead_Time', 'Median_Lead_Time', 'Std_Dev_Lead_Time',
                'CV_%', 'Efficiency_Score']

print(f"\n  Total routes: {len(route_ranked)}")
print(f"\n  Complete Ranking (by Avg Lead Time):\n")
print(route_ranked[display_cols].to_string())

# TOP 10 MOST EFFICIENT ROUTES
print("TOP 10 MOST EFFICIENT ROUTES")

top10 = route_eff_sorted.head(10).copy()
top10['Efficiency_Rank'] = range(1, 11)

print("\n  Ranked by Composite Efficiency Score (70% speed + 30% consistency):\n")
print(top10[['Efficiency_Rank', 'Region', 'Total_Shipments',
             'Avg_Lead_Time', 'Median_Lead_Time', 'Std_Dev_Lead_Time',
             'CV_%', 'Efficiency_Score', 'Profit_Margin_%']].to_string())

print("\n\n  Why these routes rank high:")
print("   Lower average lead time (faster delivery)")
print("   Lower CV% (more consistent / predictable)")
print(f"  Avg Lead Time range: {top10['Avg_Lead_Time'].min():.1f} – {top10['Avg_Lead_Time'].max():.1f} days")
print(f"  Avg Efficiency Score: {top10['Efficiency_Score'].mean():.1f}")

# BOTTOM 10 LEAST EFFICIENT ROUTES
print("BOTTOM 10 LEAST EFFICIENT ROUTES")

bottom10 = route_eff_sorted.tail(10).sort_values('Efficiency_Score', ascending=True).copy()
bottom10['Efficiency_Rank'] = range(len(route_eff_sorted), len(route_eff_sorted) - 10, -1)

print("\n  Ranked by Composite Efficiency Score (lowest = worst):\n")
print(bottom10[['Efficiency_Rank', 'Region', 'Total_Shipments',
                'Avg_Lead_Time', 'Median_Lead_Time', 'Std_Dev_Lead_Time',
                'CV_%', 'Efficiency_Score', 'Profit_Margin_%']].to_string())

print("\n  Why these routes rank low:")
print(" Higher average lead time (slower delivery)")
print(" Higher CV% (less consistent / unpredictable)")
print(f" Avg Lead Time range: {bottom10['Avg_Lead_Time'].min():.1f} – {bottom10['Avg_Lead_Time'].max():.1f} days")
print(f" Avg Efficiency Score: {bottom10['Efficiency_Score'].mean():.1f}")

# EFFICIENCY GAP ANALYSIS
print("EFFICIENCY GAP — Top 10 vs Bottom 10")

gap_data = {
    'Metric': ['Avg Lead Time (days)', 'Median Lead Time (days)',
               'Std Dev Lead Time', 'Avg CV%',
               'Avg Efficiency Score', 'Avg Profit Margin %',
               'Total Shipments'],
    'Top 10 (Best)': [
        f"{top10['Avg_Lead_Time'].mean():.1f}",
        f"{top10['Median_Lead_Time'].mean():.1f}",
        f"{top10['Std_Dev_Lead_Time'].mean():.1f}",
        f"{top10['CV_%'].mean():.1f}%",
        f"{top10['Efficiency_Score'].mean():.1f}",
        f"{top10['Profit_Margin_%'].mean():.1f}%",
        f"{top10['Total_Shipments'].sum():,}"
    ],
    'Bottom 10 (Worst)': [
        f"{bottom10['Avg_Lead_Time'].mean():.1f}",
        f"{bottom10['Median_Lead_Time'].mean():.1f}",
        f"{bottom10['Std_Dev_Lead_Time'].mean():.1f}",
        f"{bottom10['CV_%'].mean():.1f}%",
        f"{bottom10['Efficiency_Score'].mean():.1f}",
        f"{bottom10['Profit_Margin_%'].mean():.1f}%",
        f"{bottom10['Total_Shipments'].sum():,}"
    ]
}
gap_df = pd.DataFrame(gap_data)
print("\n" + gap_df.to_string(index=False))

# COMPARE PERFORMANCE ACROSS SHIP MODES
print("PERFORMANCE COMPARISON ACROSS SHIP MODES")

# Overall Ship Mode Performance
ship_perf = df.groupby('Ship Mode').agg(
    Total_Shipments    = ('Row ID', 'count'),
    Avg_Lead_Time      = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time   = ('Shipping_Lead_Time_Days', 'median'),
    Std_Dev_Lead_Time  = ('Shipping_Lead_Time_Days', 'std'),
    Min_Lead_Time      = ('Shipping_Lead_Time_Days', 'min'),
    Max_Lead_Time      = ('Shipping_Lead_Time_Days', 'max'),
    Total_Sales        = ('Sales', 'sum'),
    Total_Profit       = ('Gross Profit', 'sum'),
    Total_Units        = ('Units', 'sum')
).round(2)

ship_perf['CV_%'] = ((ship_perf['Std_Dev_Lead_Time'] / ship_perf['Avg_Lead_Time']) * 100).round(2)
ship_perf['Pct_of_Shipments'] = (ship_perf['Total_Shipments'] / len(df) * 100).round(2)
ship_perf['Profit_Margin_%']  = ((ship_perf['Total_Profit'] / ship_perf['Total_Sales']) * 100).round(2)
ship_perf['Revenue_Per_Shipment'] = (ship_perf['Total_Sales'] / ship_perf['Total_Shipments']).round(2)

# Rank ship modes by efficiency
ship_perf = ship_perf.sort_values('Avg_Lead_Time', ascending=True)
ship_perf['Ship_Mode_Rank'] = range(1, len(ship_perf) + 1)

print("\n  Ship Modes Ranked by Average Lead Time:\n")
print(ship_perf.to_string())

# --- Ship Mode × Region Cross-Tab ---
print("\n\n  Ship Mode × Region — Average Lead Time (Days):")
cross_avg = df.pivot_table(
    values='Shipping_Lead_Time_Days',
    index='Ship Mode',
    columns='Region',
    aggfunc='mean'
).round(1)
cross_avg['Overall'] = df.groupby('Ship Mode')['Shipping_Lead_Time_Days'].mean().round(1)
cross_avg = cross_avg.sort_values('Overall')
print(cross_avg.to_string())

print("\n\n  Ship Mode × Region — Shipment Count:")
cross_cnt = df.pivot_table(
    values='Row ID',
    index='Ship Mode',
    columns='Region',
    aggfunc='count',
    margins=True,
    margins_name='Total'
)
print(cross_cnt.to_string())

# --- Ship Mode × Region — CV% (Consistency) ---
print("\n\n  Ship Mode × Region — Lead Time CV% (lower = more consistent):")
cross_cv = df.groupby(['Ship Mode', 'Region'])['Shipping_Lead_Time_Days'].agg(
    lambda x: (x.std() / x.mean() * 100) if x.mean() > 0 else 0
).unstack().round(1)
print(cross_cv.to_string())

# --- Per-Route Ship Mode Breakdown (Top 10 busiest routes) ---
print("\n\n  Top 10 Busiest Routes — Ship Mode Breakdown:")
top_routes = route_ranked.head(10).index.tolist()

# Get busiest 10 routes instead
busiest_routes = route_eff.sort_values('Total_Shipments', ascending=False).head(10).index.tolist()
route_ship = df[df['Route'].isin(busiest_routes)].groupby(['Route', 'Ship Mode']).agg(
    Shipments      = ('Row ID', 'count'),
    Avg_Lead_Time  = ('Shipping_Lead_Time_Days', 'mean')
).round(1)

print(route_ship.to_string())

# --- Ship Mode share per route (pivot) ---
print("\n\n  Ship Mode Share (%) for Top 10 Busiest Routes:")
ship_share = df[df['Route'].isin(busiest_routes)].pivot_table(
    values='Row ID',
    index='Route',
    columns='Ship Mode',
    aggfunc='count',
    fill_value=0
)
ship_share_pct = ship_share.div(ship_share.sum(axis=1), axis=0).multiply(100).round(1)
print(ship_share_pct.to_string())

#  KEY FINDINGS
print(" KEY FINDINGS SUMMARY")

fastest_mode = ship_perf.index[0]
slowest_mode = ship_perf.index[-1]

print(f"""
   EFFICIENCY RANKING:
    • Total routes ranked          : {len(route_ranked)}
    • Highest Efficiency Score     : {route_eff_sorted.iloc[0].name} ({route_eff_sorted.iloc[0]['Efficiency_Score']})
    • Lowest Efficiency Score      : {route_eff_sorted.iloc[-1].name} ({route_eff_sorted.iloc[-1]['Efficiency_Score']})
    • Score Range                  : {route_eff_sorted['Efficiency_Score'].min():.1f} – {route_eff_sorted['Efficiency_Score'].max():.1f}

   TOP 10 vs BOTTOM 10 GAP:
    • Avg Lead Time Gap            : {bottom10['Avg_Lead_Time'].mean() - top10['Avg_Lead_Time'].mean():.1f} days slower
    • CV% Gap                      : {bottom10['CV_%'].mean() - top10['CV_%'].mean():.1f}% more variable
    • Efficiency Score Gap         : {top10['Efficiency_Score'].mean() - bottom10['Efficiency_Score'].mean():.1f} points

   SHIP MODE COMPARISON:
    • Fastest Mode                 : {fastest_mode} (Avg {ship_perf.loc[fastest_mode, 'Avg_Lead_Time']:.1f} days)
    • Slowest Mode                 : {slowest_mode} (Avg {ship_perf.loc[slowest_mode, 'Avg_Lead_Time']:.1f} days)
    • Most Used Mode               : {ship_perf['Total_Shipments'].idxmax()} ({ship_perf['Total_Shipments'].max():,} shipments, {ship_perf.loc[ship_perf['Total_Shipments'].idxmax(), 'Pct_of_Shipments']}%)
    • Most Consistent Mode (CV%)   : {ship_perf['CV_%'].idxmin()} (CV = {ship_perf['CV_%'].min():.1f}%)
""")




EFFICIENCY BENCHMARKING
ALL ROUTES RANKED — Fastest to Slowest

  Total routes: 59

  Complete Ranking (by Avg Lead Time):

                                                    Speed_Rank  Efficiency_Rank    Region  Total_Shipments  Avg_Lead_Time  Median_Lead_Time  Std_Dev_Lead_Time   CV_%  Efficiency_Score
Route                                                                                                                                                                                  
Nassau Candy (Factory) → Maine                               1                1  Atlantic                8        1137.12            1274.0             189.73  16.69             83.00
Nassau Candy (Factory) → Newfoundland and Labrador           2               12  Atlantic                6        1216.17            1094.5             357.45  29.39             58.95
Nassau Candy (Factory) → Nevada                              3                2   Pacific               39        1226.79            1273.0 

In [ ]:

# GEOGRAPHIC BOTTLENECK ANALYSIS
# Identify regions with high average lead time
# Identify high shipment volume + poor performance zones
# Detect congestion-prone states or regions


df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d-%m-%Y', dayfirst=True)
df['Shipping_Lead_Time_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

FACTORY = "Nassau Candy (Factory)"
overall_avg_lt = df['Shipping_Lead_Time_Days'].mean()
overall_med_lt = df['Shipping_Lead_Time_Days'].median()

print("GEOGRAPHIC BOTTLENECK ANALYSIS")
print(f"\n  Baseline — Overall Avg Lead Time : {overall_avg_lt:.2f} days")
print(f"  Baseline — Overall Median Lead Time : {overall_med_lt:.0f} days")

# REGIONAL BOTTLENECK ANALYSIS
print("REGIONAL BOTTLENECK ANALYSIS")

region_stats = df.groupby('Region').agg(
    Total_Shipments    = ('Row ID', 'count'),
    Avg_Lead_Time      = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time   = ('Shipping_Lead_Time_Days', 'median'),
    Std_Dev_Lead_Time  = ('Shipping_Lead_Time_Days', 'std'),
    Min_Lead_Time      = ('Shipping_Lead_Time_Days', 'min'),
    Max_Lead_Time      = ('Shipping_Lead_Time_Days', 'max'),
    Total_Sales        = ('Sales', 'sum'),
    Total_Profit       = ('Gross Profit', 'sum'),
    Unique_States      = ('State/Province', 'nunique'),
    Unique_Cities      = ('City', 'nunique'),
    Unique_Customers   = ('Customer ID', 'nunique')
).round(2)

region_stats['CV_%'] = ((region_stats['Std_Dev_Lead_Time'] / region_stats['Avg_Lead_Time']) * 100).round(2)
region_stats['Pct_of_Shipments'] = (region_stats['Total_Shipments'] / len(df) * 100).round(2)
region_stats['Delay_vs_Baseline'] = (region_stats['Avg_Lead_Time'] - overall_avg_lt).round(2)
region_stats['Profit_Margin_%'] = ((region_stats['Total_Profit'] / region_stats['Total_Sales']) * 100).round(2)

# Flag regions above average
region_stats['Above_Avg_Lead_Time'] = np.where(
    region_stats['Avg_Lead_Time'] > overall_avg_lt, 'YES', 'NO'
)

region_stats = region_stats.sort_values('Avg_Lead_Time', ascending=False)

print("\n  All Regions — Lead Time Performance:\n")
print(region_stats.to_string())

print(f"\n Regions with ABOVE-AVERAGE Lead Time (> {overall_avg_lt:.1f} days):")
above_avg = region_stats[region_stats['Avg_Lead_Time'] > overall_avg_lt]
if len(above_avg) > 0:
    for idx, row in above_avg.iterrows():
        print(f"   {idx}: {row['Avg_Lead_Time']:.1f} days "
              f"(+{row['Delay_vs_Baseline']:.1f} days above baseline, "
              f"{row['Total_Shipments']:,} shipments, {row['Pct_of_Shipments']}%)")
else:
    print("    None — all regions are at or below baseline")

# STATE-LEVEL BOTTLENECK DEEP DIVE
print("STATE LEVEL BOTTLENECK ANALYSIS")

state_stats = df.groupby(['State/Province', 'Region']).agg(
    Total_Shipments    = ('Row ID', 'count'),
    Avg_Lead_Time      = ('Shipping_Lead_Time_Days', 'mean'),
    Median_Lead_Time   = ('Shipping_Lead_Time_Days', 'median'),
    Std_Dev_Lead_Time  = ('Shipping_Lead_Time_Days', 'std'),
    Total_Sales        = ('Sales', 'sum'),
    Total_Profit       = ('Gross Profit', 'sum'),
    Unique_Cities      = ('City', 'nunique'),
    Unique_Customers   = ('Customer ID', 'nunique')
).round(2)

state_stats['CV_%'] = ((state_stats['Std_Dev_Lead_Time'] / state_stats['Avg_Lead_Time']) * 100).round(2)
state_stats['Pct_of_Shipments'] = (state_stats['Total_Shipments'] / len(df) * 100).round(2)
state_stats['Delay_vs_Baseline'] = (state_stats['Avg_Lead_Time'] - overall_avg_lt).round(2)

# --- States with HIGH Average Lead Time ---
print("\n  Top 15 States with HIGHEST Average Lead Time:\n")
slow_states = state_stats.sort_values('Avg_Lead_Time', ascending=False).head(15)
print(slow_states[['Total_Shipments', 'Avg_Lead_Time', 'Median_Lead_Time',
                    'Std_Dev_Lead_Time', 'CV_%', 'Delay_vs_Baseline',
                    'Pct_of_Shipments']].to_string())

# --- States with High Volume + Above-Avg Lead Time (BOTTLENECKS) ---
print("\n\n   HIGH-VOLUME BOTTLENECK STATES (above-avg lead time + ≥50 shipments):\n")
volume_threshold = 50
bottleneck_states = state_stats[
    (state_stats['Avg_Lead_Time'] > overall_avg_lt) &
    (state_stats['Total_Shipments'] >= volume_threshold)
].sort_values('Total_Shipments', ascending=False)

if len(bottleneck_states) > 0:
    print(bottleneck_states[['Total_Shipments', 'Avg_Lead_Time', 'Median_Lead_Time',
                              'Std_Dev_Lead_Time', 'CV_%', 'Delay_vs_Baseline',
                              'Pct_of_Shipments']].to_string())
    total_bottleneck_shipments = bottleneck_states['Total_Shipments'].sum()
    print(f"\n  → {len(bottleneck_states)} bottleneck states affecting "
          f"{total_bottleneck_shipments:,} shipments "
          f"({total_bottleneck_shipments/len(df)*100:.1f}% of total)")
else:
    print("  None identified")

# CONGESTION DETECTION — VOLUME-WEIGHTED DELAY SCORE
print("CONGESTION DETECTION — Volume-Weighted Delay Score")

# Congestion Score = (Avg_Lead_Time / Overall_Avg) × log(Total_Shipments)
# Higher score = more concerning (slow + high volume)
state_stats['Congestion_Score'] = (
    (state_stats['Avg_Lead_Time'] / overall_avg_lt) *
    np.log1p(state_stats['Total_Shipments'])
).round(2)

state_stats['Congestion_Rank'] = state_stats['Congestion_Score'].rank(ascending=False).astype(int)

print("\n  Congestion Score = (Avg Lead Time / Baseline) × ln(1 + Shipments)")
print("  Higher score → more congestion concern (slow delivery + high volume)\n")

# Top 15 most congestion-prone
print("  Top 15 MOST Congestion-Prone States:\n")
congested = state_stats.sort_values('Congestion_Score', ascending=False).head(15)
print(congested[['Total_Shipments', 'Avg_Lead_Time', 'Delay_vs_Baseline',
                  'CV_%', 'Congestion_Score', 'Congestion_Rank',
                  'Pct_of_Shipments']].to_string())

# Bottom 10 — least congestion
print("\n\n  Top 10 LEAST Congestion-Prone States:\n")
least_cong = state_stats.sort_values('Congestion_Score', ascending=True).head(10)
print(least_cong[['Total_Shipments', 'Avg_Lead_Time', 'Delay_vs_Baseline',
                    'CV_%', 'Congestion_Score', 'Congestion_Rank',
                    'Pct_of_Shipments']].to_string())

# CONGESTION AT REGION LEVEL
print("REGIONAL CONGESTION SUMMARY")

region_stats['Congestion_Score'] = (
    (region_stats['Avg_Lead_Time'] / overall_avg_lt) *
    np.log1p(region_stats['Total_Shipments'])
).round(2)
region_stats = region_stats.sort_values('Congestion_Score', ascending=False)

print("\n  Regions by Congestion Score:\n")
print(region_stats[['Total_Shipments', 'Pct_of_Shipments', 'Avg_Lead_Time',
                     'Delay_vs_Baseline', 'CV_%', 'Congestion_Score',
                     'Above_Avg_Lead_Time']].to_string())

# QUADRANT CLASSIFICATION
print("STATE QUADRANT CLASSIFICATION")

median_shipments = state_stats['Total_Shipments'].median()

def classify_quadrant(row):
    high_vol = row['Total_Shipments'] >= median_shipments
    slow     = row['Avg_Lead_Time'] > overall_avg_lt
    if high_vol and slow:
        return 'Q1: High Volume + Slow (CRITICAL)'
    elif high_vol and not slow:
        return 'Q2: High Volume + Fast (IDEAL)'
    elif not high_vol and slow:
        return 'Q3: Low Volume + Slow (MONITOR)'
    else:
        return 'Q4: Low Volume + Fast (NICHE)'

state_stats['Quadrant'] = state_stats.apply(classify_quadrant, axis=1)

quadrant_summary = state_stats.groupby('Quadrant').agg(
    State_Count     = ('Total_Shipments', 'count'),
    Total_Shipments = ('Total_Shipments', 'sum'),
    Avg_Lead_Time   = ('Avg_Lead_Time', 'mean'),
    Avg_CV          = ('CV_%', 'mean')
).round(2)
quadrant_summary['Pct_Shipments'] = (quadrant_summary['Total_Shipments'] / len(df) * 100).round(1)

print(f"\n  Quadrant thresholds:")
print(f"    Volume threshold  : {median_shipments:.0f} shipments (median)")
print(f"    Speed threshold   : {overall_avg_lt:.1f} days (overall avg)")

print(f"\n  Quadrant Summary:\n")
print(quadrant_summary.to_string())

# List states in each quadrant
for quad in sorted(state_stats['Quadrant'].unique()):
    states_in_quad = state_stats[state_stats['Quadrant'] == quad].sort_values(
        'Congestion_Score', ascending=False
    )
    print(f"\n\n  {quad}")
    print(f"  {'─' * 60}")
    for idx, row in states_in_quad.iterrows():
        state_name = idx[0]  # State/Province from multi-index
        region     = idx[1]  # Region
        print(f"    • {state_name:<25s} | {region:<10s} | "
              f"{row['Total_Shipments']:>5} shipments | "
              f"{row['Avg_Lead_Time']:>7.1f} days | "
              f"Congestion: {row['Congestion_Score']:.1f}")

# CITY-LEVEL HOTSPOTS (within bottleneck states)
print("CITY-LEVEL HOTSPOTS IN BOTTLENECK STATES")

if len(bottleneck_states) > 0:
    bottleneck_state_names = [idx[0] for idx in bottleneck_states.index]
    city_hotspots = df[df['State/Province'].isin(bottleneck_state_names)].groupby(
        ['State/Province', 'Region', 'City']
    ).agg(
        Shipments     = ('Row ID', 'count'),
        Avg_Lead_Time = ('Shipping_Lead_Time_Days', 'mean'),
        Std_Dev       = ('Shipping_Lead_Time_Days', 'std'),
        Total_Sales   = ('Sales', 'sum')
    ).round(2)

    city_hotspots['Delay_vs_Baseline'] = (city_hotspots['Avg_Lead_Time'] - overall_avg_lt).round(2)

    print("\n  Top 20 Cities with HIGHEST Lead Time (in bottleneck states, min 5 shipments):\n")
    filtered = city_hotspots[city_hotspots['Shipments'] >= 5].sort_values(
        'Avg_Lead_Time', ascending=False
    ).head(20)
    print(filtered.to_string())
else:
    print("\n  No bottleneck states identified — skipping city-level analysis")

# KEY FINDINGS
print("KEY FINDINGS — GEOGRAPHIC BOTTLENECK SUMMARY")

q1_count = len(state_stats[state_stats['Quadrant'].str.contains('Q1')])
q1_shipments = state_stats[state_stats['Quadrant'].str.contains('Q1')]['Total_Shipments'].sum()
most_congested = state_stats.sort_values('Congestion_Score', ascending=False).iloc[0]
most_congested_name = most_congested.name[0]

print(f"""
  GEOGRAPHIC OVERVIEW:
     Total states/provinces analyzed : {state_stats.shape[0]}
     Regions analyzed                : {region_stats.shape[0]}
     Overall Avg Lead Time baseline  : {overall_avg_lt:.1f} days

   BOTTLENECK SUMMARY:
    States above avg lead time      : {len(state_stats[state_stats['Avg_Lead_Time'] > overall_avg_lt])}
    High-volume bottleneck states   : {len(bottleneck_states)} (≥{volume_threshold} shipments + above avg lead time)
    Shipments affected              : {bottleneck_states['Total_Shipments'].sum():,} ({bottleneck_states['Total_Shipments'].sum()/len(df)*100:.1f}% of total)

  MOST CONGESTED STATE:
     {most_congested_name}
     Congestion Score : {most_congested['Congestion_Score']:.1f}
     Avg Lead Time    : {most_congested['Avg_Lead_Time']:.1f} days (+{most_congested['Delay_vs_Baseline']:.1f} vs baseline)
     Total Shipments  : {most_congested['Total_Shipments']:,}

  QUADRANT BREAKDOWN:
    Q1 Critical (High Vol + Slow) : {q1_count} states, {q1_shipments:,} shipments
    Q2 Ideal (High Vol + Fast)    : {len(state_stats[state_stats['Quadrant'].str.contains('Q2')])} states
    Q3 Monitor (Low Vol + Slow)   : {len(state_stats[state_stats['Quadrant'].str.contains('Q3')])} states
    Q4 Niche (Low Vol + Fast)     : {len(state_stats[state_stats['Quadrant'].str.contains('Q4')])} states

  RECOMMENDATIONS:
    Focus optimization on Q1 (Critical) states first — high impact
    Investigate root causes in top 5 congested states
    Consider warehouse/distribution center placement near bottleneck zones
    Monitor Q3 states — if volume grows they may become Q1
""")



GEOGRAPHIC BOTTLENECK ANALYSIS

  Baseline — Overall Avg Lead Time : 1320.84 days
  Baseline — Overall Median Lead Time : 1274 days
REGIONAL BOTTLENECK ANALYSIS

  All Regions — Lead Time Performance:

          Total_Shipments  Avg_Lead_Time  Median_Lead_Time  Std_Dev_Lead_Time  Min_Lead_Time  Max_Lead_Time  Total_Sales  Total_Profit  Unique_States  Unique_Cities  Unique_Customers   CV_%  Pct_of_Shipments  Delay_vs_Baseline  Profit_Margin_% Above_Avg_Lead_Time
Region                                                                                                                                                                                                                                                                 
Interior             2335        1323.09            1274.0             262.69            904           1642     32037.60      21282.49             14            182              1176  19.85             22.91               2.25            66.43                 YES
Atlant

In [ ]:
# SHIP MODE PERFORMANCE ANALYSIS
# Compare shipping efficiency: Standard vs Expedited
# Evaluate cost-time tradeoffs (descriptive)


df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y', dayfirst=True)
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Profit Margin (%)'] = (df['Gross Profit'] / df['Sales'] * 100).round(2)

print(" SHIP MODE PERFORMANCE ANALYSIS")

# Individual Ship Mode Performance
print("INDIVIDUAL SHIP MODE PERFORMANCE")

ship_perf = df.groupby('Ship Mode').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Median_Shipping_Days=('Shipping Days', 'median'),
    Std_Shipping_Days=('Shipping Days', 'std'),
    Min_Shipping_Days=('Shipping Days', 'min'),
    Max_Shipping_Days=('Shipping Days', 'max'),
    Avg_Sales=('Sales', 'mean'),
    Total_Sales=('Sales', 'sum'),
    Avg_Cost=('Cost', 'mean'),
    Total_Cost=('Cost', 'sum'),
    Avg_Profit=('Gross Profit', 'mean'),
    Total_Profit=('Gross Profit', 'sum'),
    Avg_Units=('Units', 'mean')
).round(2)
ship_perf['Pct_of_Orders'] = (ship_perf['Order_Count'] / len(df) * 100).round(2)
ship_perf['Profit_Margin_%'] = (ship_perf['Total_Profit'] / ship_perf['Total_Sales'] * 100).round(2)
ship_perf['CV_Shipping'] = (ship_perf['Std_Shipping_Days'] / ship_perf['Avg_Shipping_Days']).round(4)

print(ship_perf.to_string())

# Percentile analysis
print("\n\nShipping Days Percentiles by Ship Mode:")
print(f"{'Ship Mode':<18s} {'P25':>6s} {'P50':>6s} {'P75':>6s} {'P90':>6s} {'P95':>6s}")
print("-" * 52)
for mode in df['Ship Mode'].unique():
    sub = df[df['Ship Mode'] == mode]['Shipping Days']
    print(f"{mode:<18s} {sub.quantile(0.25):>6.0f} {sub.quantile(0.5):>6.0f} "
          f"{sub.quantile(0.75):>6.0f} {sub.quantile(0.9):>6.0f} {sub.quantile(0.95):>6.0f}")

# Standard vs Expedited Shipping Comparison
print("STANDARD vs EXPEDITED SHIPPING COMPARISON")

# Classify ship modes into two categories
# Standard  = Standard Class + Second Class
# Expedited = First Class + Same Day
df['Ship Category'] = df['Ship Mode'].apply(
    lambda x: 'Standard Shipping' if x in ['Standard Class', 'Second Class'] else 'Expedited Shipping'
)

cat_stats = df.groupby('Ship Category').agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Median_Shipping_Days=('Shipping Days', 'median'),
    Std_Shipping_Days=('Shipping Days', 'std'),
    Min_Shipping_Days=('Shipping Days', 'min'),
    Max_Shipping_Days=('Shipping Days', 'max'),
    Avg_Sales=('Sales', 'mean'),
    Total_Sales=('Sales', 'sum'),
    Avg_Cost=('Cost', 'mean'),
    Total_Cost=('Cost', 'sum'),
    Avg_Profit=('Gross Profit', 'mean'),
    Total_Profit=('Gross Profit', 'sum'),
    Avg_Units=('Units', 'mean'),
    Total_Units=('Units', 'sum')
).round(2)
cat_stats['Pct_of_Orders'] = (cat_stats['Order_Count'] / len(df) * 100).round(2)
cat_stats['Profit_Margin_%'] = (cat_stats['Total_Profit'] / cat_stats['Total_Sales'] * 100).round(2)
cat_stats['CV_Shipping'] = (cat_stats['Std_Shipping_Days'] / cat_stats['Avg_Shipping_Days']).round(4)

print(cat_stats.to_string())

# Compute deltas
std_row = cat_stats.loc['Standard Shipping']
exp_row = cat_stats.loc['Expedited Shipping']
day_diff = exp_row['Avg_Shipping_Days'] - std_row['Avg_Shipping_Days']
cost_diff = exp_row['Avg_Cost'] - std_row['Avg_Cost']
profit_diff = exp_row['Avg_Profit'] - std_row['Avg_Profit']
margin_diff = exp_row['Profit_Margin_%'] - std_row['Profit_Margin_%']

print(f"\n  Expedited vs Standard Delta:")
print(f"    Avg Shipping Days:  {day_diff:+.2f} days ({'faster' if day_diff < 0 else 'slower'})")
print(f"    Avg Cost/Order:     ${cost_diff:+.2f}")
print(f"    Avg Profit/Order:   ${profit_diff:+.2f}")
print(f"    Profit Margin:      {margin_diff:+.2f} pp")

# Ship Category Performance by Region
print("SHIP CATEGORY PERFORMANCE BY REGION")

cat_region = df.groupby(['Ship Category', 'Region']).agg(
    Order_Count=('Row ID', 'count'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Avg_Cost=('Cost', 'mean'),
    Avg_Profit=('Gross Profit', 'mean')
).round(2)
print(cat_region.to_string())

print("\n Ship Category Distribution by Region (%):")
cross_cat = pd.crosstab(df['Region'], df['Ship Category'], normalize='index').round(4) * 100
print(cross_cat.to_string())

# Cost-Time Tradeoff Analysis (Descriptive)

print("COST-TIME TRADEOFF ANALYSIS (Descriptive)")

# Cost/Sales/Profit per shipping day — efficiency per day in transit
df['Cost_Per_Ship_Day'] = (df['Cost'] / df['Shipping Days']).round(4)
df['Sales_Per_Ship_Day'] = (df['Sales'] / df['Shipping Days']).round(4)
df['Profit_Per_Ship_Day'] = (df['Gross Profit'] / df['Shipping Days']).round(4)

tradeoff = df.groupby('Ship Mode').agg(
    Avg_Cost_Per_Ship_Day=('Cost_Per_Ship_Day', 'mean'),
    Avg_Sales_Per_Ship_Day=('Sales_Per_Ship_Day', 'mean'),
    Avg_Profit_Per_Ship_Day=('Profit_Per_Ship_Day', 'mean'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Avg_Cost=('Cost', 'mean'),
    Avg_Profit=('Gross Profit', 'mean')
).round(4)
tradeoff = tradeoff.sort_values('Avg_Shipping_Days')

print("\nCost-Time Efficiency Metrics by Ship Mode:")
print(tradeoff.to_string())

# Category-level tradeoff
tradeoff_cat = df.groupby('Ship Category').agg(
    Avg_Cost_Per_Ship_Day=('Cost_Per_Ship_Day', 'mean'),
    Avg_Sales_Per_Ship_Day=('Sales_Per_Ship_Day', 'mean'),
    Avg_Profit_Per_Ship_Day=('Profit_Per_Ship_Day', 'mean'),
    Avg_Shipping_Days=('Shipping Days', 'mean'),
    Avg_Cost=('Cost', 'mean'),
    Avg_Profit=('Gross Profit', 'mean')
).round(4)
print("\n\nCost-Time Efficiency: Standard vs Expedited:")
print(tradeoff_cat.to_string())

# Cost-Time Tradeoff — Key Observations

print("6E. COST-TIME TRADEOFF — KEY OBSERVATIONS")

fastest_mode = tradeoff.index[0]
slowest_mode = tradeoff.index[-1]

print(f"""
  SHIPPING SPEED COMPARISON:
    - Fastest mode (by avg days):  {fastest_mode} ({tradeoff.loc[fastest_mode, 'Avg_Shipping_Days']:.1f} days avg)
    - Slowest mode (by avg days):  {slowest_mode} ({tradeoff.loc[slowest_mode, 'Avg_Shipping_Days']:.1f} days avg)
    - Speed gap:                   {tradeoff.loc[slowest_mode, 'Avg_Shipping_Days'] - tradeoff.loc[fastest_mode, 'Avg_Shipping_Days']:.1f} days

  COST-TIME TRADEOFF:
    - Expedited avg cost/order:    ${exp_row['Avg_Cost']:.2f}
    - Standard avg cost/order:     ${std_row['Avg_Cost']:.2f}
    - Cost premium for expedited:  ${abs(cost_diff):.2f} {'more' if cost_diff > 0 else 'less'} per order
    - Days saved with expedited:   {abs(day_diff):.1f} days {'faster' if day_diff < 0 else 'slower'}

  PROFITABILITY IMPACT:
    - Standard profit margin:      {std_row['Profit_Margin_%']:.2f}%
    - Expedited profit margin:     {exp_row['Profit_Margin_%']:.2f}%
    - Margin difference:           {abs(margin_diff):.2f} pp {'higher' if margin_diff > 0 else 'lower'} for expedited

  DELIVERY CONSISTENCY:
    - Standard CV (lower=better):  {std_row['CV_Shipping']:.4f}
    - Expedited CV (lower=better): {exp_row['CV_Shipping']:.4f}
    - {'Expedited' if exp_row['CV_Shipping'] < std_row['CV_Shipping'] else 'Standard'} shipping is MORE consistent

  ORDER VOLUME SPLIT:
    - Standard shipping:           {std_row['Pct_of_Orders']:.1f}% of all orders ({int(std_row['Order_Count']):,} orders)
    - Expedited shipping:          {exp_row['Pct_of_Orders']:.1f}% of all orders ({int(exp_row['Order_Count']):,} orders)

  COST-TIME VERDICT:
    - The shipping days across modes are remarkably similar (range ~{tradeoff['Avg_Shipping_Days'].max() - tradeoff['Avg_Shipping_Days'].min():.0f} days difference).
    - {'Expedited shipping provides marginal speed improvement at a lower cost.' if day_diff < 0 and cost_diff < 0 else 'Expedited shipping provides marginal speed improvement but at slightly higher cost.' if day_diff < 0 and cost_diff >= 0 else 'There is minimal practical difference in shipping speed between the modes.'}
    - Standard shipping dominates order volume ({std_row['Pct_of_Orders']:.0f}%), suggesting customers/operations default to it.
    - Profit margins are nearly identical across modes (~66.5%), indicating ship mode choice does not significantly impact profitability.
""")




 SHIP MODE PERFORMANCE ANALYSIS
INDIVIDUAL SHIP MODE PERFORMANCE
                Order_Count  Avg_Shipping_Days  Median_Shipping_Days  Std_Shipping_Days  Min_Shipping_Days  Max_Shipping_Days  Avg_Sales  Total_Sales  Avg_Cost  Total_Cost  Avg_Profit  Total_Profit  Avg_Units  Pct_of_Orders  Profit_Margin_%  CV_Shipping
Ship Mode                                                                                                                                                                                                                                                    
First Class            1548            1338.28                1272.0             265.63                905               1638      13.77     21319.39      4.72     7308.30        9.05      14011.09       3.70          15.19            65.72       0.1985
Same Day                547            1333.44                1269.0             253.81                904               1636      13.00      7113.67      4.41     2412.94  

In [ ]:
# EXPORT PREPROCESSED DATASET FOR POWER BI

# --- Date Parsing ---
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y', dayfirst=True)

# --- KPI 1: Shipping Lead Time (Ship Date - Order Date) ---
df['Shipping Lead Time'] = (df['Ship Date'] - df['Order Date']).dt.days

# --- Time Components ---
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Quarter'] = df['Order Date'].dt.quarter
df['Order Month Name'] = df['Order Date'].dt.month_name()

# --- Route Identifiers ---
df['Route'] = df['Region'] + ' - ' + df['State/Province'] + ' - ' + df['City']
df['Route_Region_State'] = df['Region'] + ' - ' + df['State/Province']

# --- Financial Metrics ---
df['Profit Margin (%)'] = (df['Gross Profit'] / df['Sales'] * 100).round(2)
df['Cost Per Unit'] = (df['Cost'] / df['Units']).round(2)
df['Revenue Per Unit'] = (df['Sales'] / df['Units']).round(2)

# --- KPI 4: Delay Flag (threshold = 1000 days, adjust as needed) ---
DELAY_THRESHOLD = 1000
df['Delay Flag'] = (df['Shipping Lead Time'] > DELAY_THRESHOLD).astype(int)
df['Delay Status'] = df['Delay Flag'].map({1: 'Delayed', 0: 'On-Time'})

# --- KPI 5: Route Efficiency Score (precomputed per route) ---
route_avg = df.groupby('Route_Region_State')['Shipping Lead Time'].mean()
max_avg = route_avg.max()
route_efficiency = ((1 - route_avg / max_avg) * 100).round(2)
route_efficiency.name = 'Route Efficiency Score'
df = df.merge(route_efficiency, on='Route_Region_State', how='left')

# --- Ship Category (Standard vs Expedited) ---
df['Ship Category'] = df['Ship Mode'].apply(
    lambda x: 'Standard' if x in ['Standard Class', 'Second Class'] else 'Expedited'
)

# --- Summary ---

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"\nAll Columns:\n{list(df.columns)}")
print(f"\nNew Columns Added:")
print("  - Shipping Lead Time (days)")
print("  - Order Year / Month / Quarter / Month Name")
print("  - Route (Region - State - City)")
print("  - Route_Region_State (Region - State)")
print("  - Profit Margin (%)")
print("  - Cost Per Unit / Revenue Per Unit")
print(f"  - Delay Flag (threshold = {DELAY_THRESHOLD} days)")
print("  - Delay Status (Delayed / On-Time)")
print("  - Route Efficiency Score (0-100)")
print("  - Ship Category (Standard / Expedited)")
print(f"\nSample (first 5 rows):")
print(df.head().to_string())

# --- Export ---
output_file = "Nassau_Candy_Preprocessed.csv"
df.to_csv(output_file, index=False)
print(f"\n Exported to: {output_file}")
print(f"   File size: {df.shape[0]} rows x {df.shape[1]} columns")

# --- Download in Colab ---
try:
    from google.colab import files
    files.download(output_file)
    print("Download triggered!")
except ImportError:
    print(f"Not running in Colab. File saved locally as '{output_file}'")


Rows: 10194, Columns: 43

All Columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost', 'Shipping Days', 'Order Year', 'Order Month', 'Order Quarter', 'Route', 'Route_Region_State', 'Profit Margin (%)', 'Cost Per Unit', 'Delay_Bucket', 'Shipping_Lead_Time_Days', 'Lead_Time_Category', 'Route_Factory_to_Region', 'Route_Factory_to_State', 'Route_to_Region', 'Route_to_State', 'Ship Category', 'Cost_Per_Ship_Day', 'Sales_Per_Ship_Day', 'Profit_Per_Ship_Day', 'Shipping Lead Time', 'Order Month Name', 'Revenue Per Unit', 'Delay Flag', 'Delay Status', 'Route Efficiency Score']

New Columns Added:
  - Shipping Lead Time (days)
  - Order Year / Month / Quarter / Month Name
  - Route (Region - State - City)
  - Route_Region_State (Region - State)
  - Profit Margin (%)
  - Cost Per Unit / Revenue Per Unit
  - Delay Fla

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered!
